# Notebook 01: Data Understanding & Raw Dataset Audit
### RetailIQ — Demand Forecasting & Multi-Tool Business Assistant

**Objective:** Inspect raw retail datasets (`train.csv`, `features.csv`, `stores.csv`, `test.csv`), assess data types, identify missing values, verify time ranges, evaluate store-department granularity, and audit customer return records (negative sales).


In [1]:
import sys, os
from pathlib import Path
# Add project root to path for src imports
project_root = str(Path(os.path.abspath('')).resolve())
if not os.path.exists(os.path.join(project_root, 'src')):
    project_root = str(Path(os.path.abspath('')).resolve().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import pandas as pd
import numpy as np
from src.data.loader import DataLoader

# Initialize DataLoader
loader = DataLoader()
audit_results = loader.audit_datasets()

# Convert audit to DataFrame for display
audit_summary = []
for dataset_name, stats in audit_results.items():
    audit_summary.append({
        "Dataset": dataset_name,
        "Rows": f"{stats['rows']:,}",
        "Columns": stats['columns'],
        "Date Range": f"{stats.get('date_min', 'N/A')} to {stats.get('date_max', 'N/A')}",
        "Unique Stores": stats.get('unique_stores', 'N/A'),
        "Unique Depts": stats.get('unique_departments', 'N/A'),
        "Memory (MB)": stats['memory_usage_mb']
    })

pd.DataFrame(audit_summary)


,Dataset,Rows,Columns,Date Range,Unique Stores,Unique Depts,Memory (MB)
0,stores,45,3,N/A to N/A,45,N/A,0.00
1,features,"8,190",12,2010-02-05 to 2013-07-26,45,N/A,0.70
2,train,"421,570",5,2010-02-05 to 2012-10-26,45,81,13.27
3,test,"115,064",4,2012-11-02 to 2013-07-26,45,81,2.74


### Detailed Target Inspection: `Weekly_Sales`
We inspect whether `Weekly_Sales` is available, verify summary statistics, and audit the presence of negative values representing customer returns.


In [2]:
train_df = loader.load_train()
print("Weekly_Sales Target Summary:")
print(train_df["Weekly_Sales"].describe().to_string())

neg_count = (train_df["Weekly_Sales"] < 0).sum()
neg_pct = (train_df["Weekly_Sales"] < 0).mean() * 100
print(f"\nNegative Weekly Sales Count: {neg_count:,} ({neg_pct:.3f}%)")
print(f"Minimum Value: ${train_df['Weekly_Sales'].min():,.2f}")


Weekly_Sales Target Summary:
count    421570.000000
mean      15981.258123
std       22711.183519
min       -4988.940000
25%        2079.650000
50%        7612.030000
75%       20205.852500
max      693099.360000

Negative Weekly Sales Count: 1,285 (0.305%)
Minimum Value: $-4,988.94


**Observation & Findings:**
1. The training dataset spans **421,570 weekly observations** across 45 stores and 81 departments from **2010-02-05 to 2012-10-26** (143 weeks).
2. The `Weekly_Sales` target is fully present. Exactly **1,285 rows (0.305%)** contain negative values.
3. As defined in the RetailIQ policy documentation, negative sales represent customer returns exceeding weekly gross sales. These will be preserved with a dedicated `returns_flag = 1`.
4. No product description text exists in the dataset; departments are designated by numerical IDs (1 to 99).
